# Exploration du jeu NASA C-MAPSS FD001
**Partie 1 - Mouad AAMER**

Objectifs : comprendre les donnees, calculer le label RUL, reperer les capteurs inutiles (constants) avant l'ingestion.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

COLUMNS = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f's{i}' for i in range(1, 22)]

def load(path):
    df = pd.read_csv(path, sep=r'\s+', header=None)
    df = df.iloc[:, :len(COLUMNS)]
    df.columns = COLUMNS
    return df

train = load('../data/raw/train_FD001.txt')
test = load('../data/raw/test_FD001.txt')
rul_test = pd.read_csv('../data/raw/RUL_FD001.txt', header=None, names=['RUL'])
print('train :', train.shape, '| test :', test.shape, '| RUL test :', rul_test.shape)
train.head()

## 1. Duree de vie des moteurs (train)
Chaque moteur du jeu d'entrainement est suivi jusqu'a la panne : son dernier cycle = sa duree de vie.

In [ ]:
life = train.groupby('engine_id')['cycle'].max()
print(life.describe())
life.hist(bins=20)
plt.xlabel('cycles avant la panne'); plt.ylabel('nombre de moteurs'); plt.title('Duree de vie - FD001 train')
plt.show()

## 2. Capteurs constants
Un capteur dont l'ecart-type est quasi nul n'apporte aucune information -> a supprimer en zone Silver.

In [ ]:
std = train[COLUMNS[2:]].std().sort_values()
print(std)
constant_cols = std[std < 1e-3].index.tolist()
print('\nColonnes quasi constantes :', constant_cols)

## 3. Label RUL
RUL = cycle max du moteur - cycle courant. On la plafonne a 125 : au debut de vie le moteur est sain et sa degradation n'est pas encore visible.

In [ ]:
train['RUL'] = train.groupby('engine_id')['cycle'].transform('max') - train['cycle']
train['RUL_cap'] = train['RUL'].clip(upper=125)

def etat(rul):
    if rul <= 15: return 'Critique'
    if rul <= 50: return 'A surveiller'
    return 'Sain'

train['etat'] = train['RUL'].apply(etat)
print(train['etat'].value_counts())

## 4. Capteurs les plus lies a la RUL

In [ ]:
useful = [c for c in COLUMNS[5:] if c not in constant_cols]
corr = train[useful + ['RUL_cap']].corr()['RUL_cap'].drop('RUL_cap').sort_values()
print(corr)
corr.plot.barh(figsize=(6, 5), title='Correlation avec la RUL (plafonnee)')
plt.show()

## 5. Evolution des capteurs pour quelques moteurs

In [ ]:
top = corr.abs().sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(len(top), 1, figsize=(9, 9), sharex=True)
for ax, s in zip(axes, top):
    for e in [1, 2, 3, 4]:
        d = train[train.engine_id == e]
        ax.plot(d['RUL'], d[s], label=f'moteur {e}')
    ax.set_ylabel(s); ax.invert_xaxis()
axes[0].legend(); axes[-1].set_xlabel('RUL (cycles restants)')
plt.tight_layout(); plt.show()

## Conclusions (a completer)
- Capteurs a supprimer : ...
- Capteurs les plus informatifs : ...
- Choix du plafond RUL = 125 et des seuils d'etat (15 / 50 cycles) : ...